In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

# Define the path to the best weights from the training run
trained_model_path = '/content/runs/detect/train-8/weights/best.pt'

# Load the trained model
model = YOLO(trained_model_path)

print(f"Loaded model from: {trained_model_path}")

# Path to the validation images directory, using the established new_dataset_base_path
val_images_dir = os.path.join('/content/crf-dataset-fresh', 'crf_val', 'images')

# Get a different sample image from the validation set (e.g., the second one alphabetically)
sample_image_name = sorted(os.listdir(val_images_dir))[1] # Pick the second image alphabetically
sample_image_path = os.path.join(val_images_dir, sample_image_name)

print(f"Running inference on sample image: {sample_image_path}")

# Run inference
# The 'save=True' argument will save the image with predictions overlaid
results = model.predict(source=sample_image_path, save=True, imgsz=1024, conf=0.5, name='predict_sample_2')

# Ultralytics often saves results as .jpg even if input was .png
output_image_path = os.path.join(results[0].save_dir, os.path.splitext(sample_image_name)[0] + '.jpg')

if os.path.exists(output_image_path):
    print(f"\n--- Inference Results on {sample_image_name} ---")
    display(Image(filename=output_image_path))
else:
    print(f"Predicted image not found at: {output_image_path}")

# Optional: Print detected bounding boxes and classes
print("\n--- Detections ---")
for r in results:
    for box in r.boxes:
        c = int(box.cls)
        conf = box.conf.item()
        xyxy = box.xyxy[0].tolist() # [x1, y1, x2, y2]
        print(f"Class: {model.names[c]}, Confidence: {conf:.2f}, Bounding Box: {xyxy}")

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os

# Define the path to the best weights from the training run
trained_model_path = '/content/runs/detect/train-8/weights/best.pt'

# Load the trained model
model = YOLO(trained_model_path)

print(f"Loaded model from: {trained_model_path}")

# Path to the validation images directory, using the established new_dataset_base_path
val_images_dir = os.path.join('/content/crf-dataset-fresh', 'crf_val', 'images')

# Get a different sample image from the validation set (e.g., the third one alphabetically)
sorted_val_images = sorted(os.listdir(val_images_dir))

# Ensure there are enough images to pick the third one
if len(sorted_val_images) >= 3:
    sample_image_name = sorted_val_images[2] # Pick the third image alphabetically
else:
    print("Warning: Not enough images in validation set to pick the third one. Picking the first available.")
    sample_image_name = sorted_val_images[0]

sample_image_path = os.path.join(val_images_dir, sample_image_name)

print(f"Running inference on sample image: {sample_image_path}")

# Run inference
# The 'save=True' argument will save the image with predictions overlaid
# Use a new name for the prediction run to avoid overwriting previous results
results = model.predict(source=sample_image_path, save=True, imgsz=1024, conf=0.5, name='predict_sample_3')

# Ultralytics often saves results as .jpg even if input was .png
output_image_path = os.path.join(results[0].save_dir, os.path.splitext(sample_image_name)[0] + '.jpg')

if os.path.exists(output_image_path):
    print(f"\n--- Inference Results on {sample_image_name} ---")
    display(Image(filename=output_image_path))
else:
    print(f"Predicted image not found at: {output_image_path}")

# Optional: Print detected bounding boxes and classes
print("\n--- Detections ---")
for r in results:
    for box in r.boxes:
        c = int(box.cls)
        conf = box.conf.item()
        xyxy = box.xyxy[0].tolist() # [x1, y1, x2, y2]
        print(f"Class: {model.names[c]}, Confidence: {conf:.2f}, Bounding Box: {xyxy}")

### Exporting the Trained Model

Now that we have a trained and tested model, let's export it to a common deployment format like ONNX. This allows for easier integration into various applications and environments.

In [ ]:
from ultralytics import YOLO
import os

# Define the path to the best weights from the training run
trained_model_path = '/content/runs/detect/train-8/weights/best.pt'

# Load the trained model
model = YOLO(trained_model_path)

print(f"Loaded model from: {trained_model_path}")

# Export the model to ONNX format
# You can change the 'format' argument to 'tflite', 'torchscript', 'openvino', 'coreml', etc.
exported_model_path = model.export(format='onnx')

print(f"Model successfully exported to: {exported_model_path}")

# Verify the exported file exists
if os.path.exists(exported_model_path):
    print(f"Exported model file found at: {exported_model_path}")
    print(f"File size: {os.path.getsize(exported_model_path) / (1024*1024):.2f} MB")
else:
    print(f"Error: Exported model file not found at: {exported_model_path}")

In [ ]:
Exporting to TensorRT for Jetson J40

For deployment on NVIDIA Jetson devices like the J40, converting the model to TensorRT format is crucial for optimal performance. TensorRT is NVIDIA's platform for high-performance deep learning inference, which can significantly accelerate inference speed and reduce memory footprint by optimizing the model graph for NVIDIA GPUs.

In [ ]:
from ultralytics import YOLO
import os

# Define the path to the best weights from the training run
trained_model_path = '/content/runs/detect/train-8/weights/best.pt'

# Load the trained model
model = YOLO(trained_model_path)

print(f"Loaded model from: {trained_model_path}")

# Export the model to TensorRT format (.engine file)
# The 'engine' format in Ultralytics automatically handles TensorRT conversion.
print("\nExporting model to TensorRT format...")
trt_model_path = model.export(format='engine', imgsz=1024, half=True) # half=True for FP16 precision

print(f"Model successfully exported to TensorRT at: {trt_model_path}")

# Verify the exported file exists
if os.path.exists(trt_model_path):
    print(f"Exported TensorRT model file found at: {trt_model_path}")
    print(f"File size: {os.path.getsize(trt_model_path) / (1024*1024):.2f} MB")
else:
    print(f"Error: Exported TensorRT model file not found at: {trt_model_path}")

Verify TensorRT Export with Inference

Now that the model is exported to TensorRT format, let's load it and run inference on a sample image to ensure it works correctly and produces expected detections.